In [1]:
import asyncio
import os
from pathlib import Path
from abc import ABC, abstractmethod

In [9]:
class BaseDocumentLoader(ABC):
    def __init__(self, path: str | Path):
        self.path = Path(path)

    @abstractmethod
    async def load(self) -> str:
        pass


# Reads .txt file format
class TextLoader(BaseDocumentLoader):

    async def load(self) -> str:
        #logger.info(f"Loading text file: {self.path.name}")
        if not self.path.exists():
            #logger.error(f"File not found: {self.path.name}")
            raise FileNotFoundError(f"File not found: {self.path.name}")

        loop = asyncio.get_event_loop()

        def _read_file():
            try:
                with open(self.path, "r", encoding="utf-8") as f:
                    return f.read()
            except Exception as e:
                #logger.error(f"Error reading text file {self.path.name}: {e}")
                raise

        return await loop.run_in_executor(None, _read_file)

# Reads .md file format
class MarkdownLoader(BaseDocumentLoader):
    """
    Loads markdown files (.md).
    """

    async def load(self) -> str:
        #logger.info(f"Loading markdown file: {self.path.name}")
        if not self.path.exists():
            #logger.error(f"File not found: {self.path.name}")
            raise FileNotFoundError(f"File not found: {self.path.name}")

        loop = asyncio.get_event_loop()

        def _read_file():
            try:
                with open(self.path, "r", encoding="utf-8") as f:
                    # Returns raw markdown. You could add logic here to strip markdown syntax if needed.
                    return f.read()
            except Exception as e:
                #logger.error(f"Error reading markdown file {self.path.name}: {e}")
                raise

        return await loop.run_in_executor(None, _read_file)

In [10]:
class DocumentLoadFactory:

    @staticmethod
    def get_loader(file_path: str | Path) -> BaseDocumentLoader:
        path = Path(file_path)
        ext = path.suffix.lower()  # extract extention from a path .txt, .md, .doc

        #logger.info(f"Creating loader for file: {path.name} with extension: {ext}")

        if ext == ".txt":
            return TextLoader(path)

        if ext == ".md":
            return MarkdownLoader(path)

        #logger.error(f"Unsupported file extension: {ext} for file {path.name}")
        raise ValueError(f"Unsupported file extension: {ext}")


In [14]:
class DocumentIngestionPipeline:

    def __init__(self, chunker=None, embedder=None, vector_store=None):
            print("Initializing DocumentIngestionPipeline")
        #logger.info("Initializing DocumentIngestionPipeline")
        #self.chunker = chunker or SlidingWindowChunking(
       #     chunk_size=settings.CHUNK_SIZE, overlap=settings.CHUNK_OVERLAP        )
        #self.embedder = embedder or ModelSelector
        # Configure Vector Store from settings via Factory
        #self.vector_store = vector_store or VectorStoreFactory.get_vector_store()

    async def ingest_file(self, path) -> list[str]:
        try:
            path_obj = Path(path)
            file_name = path_obj.name
            file_type = path_obj.suffix.lstrip(".")

            #logger.info(f"1. Loading Document.....{file_name}")
            loader = DocumentLoadFactory.get_loader(path_obj)
            text_content = await loader.load()
            print(text_content)
            return text_content
        
        except Exception as e:
            #logger.error(f"Failed to ingest file {file_name}: {e}")
            raise

    async def ingest_directory(self, path) -> list[str]:
        try:
            path = Path(path)
            all_files = []
            files = path.rglob("*") # reads all files in the directory and subdirectories and returns an iterable.

            for file in files:
                if file.is_file(): #ignores directories and only processes files
                    all_files.append(await self.ingest_file(file))

            return all_files
        except Exception as e:
            #ogger.error(f"Failed to ingest directory {path}: {e}")
            raise

In [18]:
pipeline = DocumentIngestionPipeline()
await pipeline.ingest_directory(r"C:\Users\prahn\OneDrive\Desktop\test_docs")

Initializing DocumentIngestionPipeline


1. Which strategy won, and on what dimension? (Accuracy?  
Parse rate? Cost?)

- All the strategies achieved a 100% parse rate. Given the limited data set, there were no scenarios this failed. For this assignment, parse rate is not a useful criteria for analysis.   
- As expected from a simple dataset, all strategies performed comparably on accuracy which is a deterministic metric in this exercise. Upon manual verification of instances of failure, the following were observed.
	- Extraction of company name is easy for all these strategies. Compared to zeroshot, the other methods did not provide any significant improvement. All the prompts performed equally well on semantics. The errors were related to punctuation and whitespaces. When the code is executed multiple times, I noticed that COT prompt that had zero errors on extracting company name, failed twice by including punctuations. While semantically this is not an error, the deterministic accu

['\n\n1. Which strategy won, and on what dimension? (Accuracy?  \nParse rate? Cost?)\n\n- All the strategies achieved a 100% parse rate. Given the limited data set, there were no scenarios this failed. For this assignment, parse rate is not a useful criteria for analysis.   \n- As expected from a simple dataset, all strategies performed comparably on accuracy which is a deterministic metric in this exercise. Upon manual verification of instances of failure, the following were observed.\n\t- Extraction of company name is easy for all these strategies. Compared to zeroshot, the other methods did not provide any significant improvement. All the prompts performed equally well on semantics. The errors were related to punctuation and whitespaces. When the code is executed multiple times, I noticed that COT prompt that had zero errors on extracting company name, failed twice by including punctuations. While semantically this is not an error, the deterministic accuracy metric chosen for this a